# Change detectors

A change detector in Skchange is a Scikit-learn-style estimator that finds *changepoints* in a single univariate or multivariate time series.
A changepoint is a point in the time series where the data distribution changes in some way.
The workflow of change detectors is familiar to anyone who has used Scikit-learn:
You first initialise the detector with algorithm-specific parameters, before calling `fit` on a time series to fit the detector on training data.
Finally, you can use the fitted detector to detect changepoints on new data with `predict`.

`predict(X)`
: Returns a numpy array of changepoint indices `cps` of shape `(n_changepoints,)`. Each changepoint marks the *inclusive start of a new segment*, so the segments of the input time series are `X[:cps[0]]`, `X[cps[0]:cps[1]]`, ..., `X[cps[-1]:]`.

As a concrete example, the snippet below generates a univariate series with a single mean change at index 20, and uses [PELT](../../api_reference/auto_generated/skchange.new_api.detectors.PELT.rst) to detect changepoints. `fit_predict` is the standard shortcut for `fit(X).predict(X)`.

In [ ]:
from skchange.new_api.datasets import generate_piecewise_normal_data
from skchange.new_api.detectors import PELT
from skchange.new_api.interval_scorers import L2Cost

# A univariate series with a change in mean at index 20.
X = generate_piecewise_normal_data(means=[0, 5], lengths=[20, 10], seed=1)

detector = PELT(cost=L2Cost(), penalty=10.0)
detector.fit_predict(X)

## Segment anomaly detectors
Some change detectors in Skchange can natively identify *anomalous segments*, meaning segments where the data deviates from a "normal" or "baseline" behaviour.
In this case, the changepoints are the boundaries of the anomalous segments.
Such detectors expose the following additional method:

`predict_segment_anomalies(X)`
: Returns a numpy array of anomalous intervals of shape `(n_anomalies, 2)`, where each row holds the start (inclusive) and end (exclusive) of a segment anomaly. The samples outside these intervals are considered normal.

For example, running [CAPA](../../api_reference/auto_generated/skchange.new_api.detectors.CAPA.rst) on the same `X` returns the post-change region as a single anomalous segment relative to the baseline mean learned robustly from the data:

In [ ]:
from skchange.new_api.detectors import CAPA
from skchange.new_api.interval_scorers import L2Saving

anomaly_detector = CAPA(segment_saving=L2Saving(), segment_penalty=10.0).fit(X)
anomaly_detector.predict_segment_anomalies(X)

## Advanced outputs
Most Skchange detectors also expose the following methods for advanced users:

`predict_scores(X, return_index=False)`
: Returns the detector's internal interval scoring objective as a 1D numpy array. The length depends on the algorithm and is not generally equal to `n_samples`. With `return_index=True` it returns a `(scores, index_dict)` tuple, where `index_dict` carries algorithm-specific metadata that locates each score on the input timeline. Used by the tuning module for penalty calibration.

`predict_all(X)`
: Convenience method on detectors that compute all outputs in a single pass, which could be more than the sum of the individual methods. Returns a dict whose keys are detector-specific. Useful for power users who want all the details in one go, without repeating work.

## Scikit-learn compatibility

All Skchange detectors inherit from scikit-learn's `BaseEstimator` via Skchange's [BaseChangeDetector](../../api_reference/auto_generated/skchange.new_api.detectors.BaseChangeDetector.rst) class, and the aim is to follow the Scikit-learn API conventions as closely as possible. This gives you sklearn-standard machinery for free, but a few sklearn tools are intentionally not supported because they assume properties that time series data do not have.

**Data types:** Like Scikit-learn estimators, Skchange detectors accept 2D array-like input of shape `(n_samples, n_features)` and return an `np.ndarray` (except the `predict_all` method intended for advanced use).

**What works:**

- `get_params` / `set_params` for inspecting and updating hyperparameters.
- `sklearn.base.clone` for making unfitted copies.
- `Pipeline` with sklearn transformers that don't rely on the ordering of samples.
- Fitted-attribute convention: attributes set in `fit` end with `_` (e.g. `detector.penalty_`), and `check_is_fitted` works as expected.

**What does not work, and why:**

- `GridSearchCV` / `cross_val_score` and other cross-validation utilities. Scikit-learn's cross-validation tools don't respect the ordering of samples crucial to time series, and thus cannot be used with Skchange detectors. Use the built-in tuning utilities in [skchange.new_api.tuning](../../api_reference/tuning.rst) instead.